# 01 - Data Understanding

## 1. Introduction

### Background

Middendorf Yoga is a yoga studio in Berlin that uses an online booking system for members to reserve places in yoga classes. Members can book or cancel a reservation at any time before the class begins, resulting in a continuously changing number of registered participants.

In practice, attendance often differs substantially from the number of active bookings. Some members cancel shortly before the start of a class, while others rarely cancel and attend with remarkable consistency. As a result, the number of participants expected from the booking data alone can be a poor estimate of the actual attendance.

The goal of this project is to predict class attendance before the class takes place—ideally several days in advance, or at least 24 hours before the scheduled start. Reliable attendance predictions can help instructors prepare more effectively and improve the overall experience for members by providing a better estimate of expected class occupancy.

### Available Datasets

The analyses presented in this project are based on two complementary datasets collected from the studio’s online booking system.

The booking events dataset records every booking and cancellation event occurring during the seven days preceding a class, up to the scheduled class start. Each event captures the updated booking and waiting lists after the action, together with contextual information such as the class, instructor, studio location, and whether the corresponding week contained a studio holiday. A studio holiday refers to any day on which all scheduled classes were cancelled by the studio, for example due to public holidays or planned studio closures. Since cancellations or closures often influence booking behavior in surrounding classes, this information is included as a potential predictive feature.

The attendance dataset records the final attendance outcome for each class. It contains the list of members who actually attended, the final waiting list at class start, the maximum class capacity, and the associated class and instructor information. This dataset represents the ground truth used to train and evaluate attendance prediction models.

### Objective of this notebook

The objective of this notebook is to obtain an initial understanding of the available datasets. We examine their structure, assess data quality, and identify the most important characteristics of the raw data. The insights gained here provide the foundation for the preprocessing and modeling steps developed in subsequent notebooks.

## 2. Data sources

The original datasets used during the development of this project contain personally identifiable information and therefore cannot be published as part of this repository.

To make the complete workflow reproducible, the repository includes a synthetic dataset that reproduces the structure and data types of the original data while containing no real member information. The synthetic data are intended solely for demonstration and testing purposes and allow every notebook in this repository to be executed without access to the production data.

Unless stated otherwise, all analyses, observations, and conclusions discussed throughout this repository refer to the original studio datasets. The synthetic dataset is provided only to demonstrate the implementation and is not expected to reproduce the statistical characteristics of the original data.

In [ ]:
import pandas as pd
from src.data import load_data, prepare_attendance, prepare_booking_events

USE_SYNTHETIC = False

booking_events_raw, attendance_raw = load_data(use_synthetic=USE_SYNTHETIC)


### 2.2 Create canonical representations

In [ ]:
booking_events = prepare_booking_events(booking_events_raw)

attendance = prepare_attendance(attendance_raw)

## 3. Dataset structure

This section examines the structure of the two datasets, including their dimensions, columns and data types.

In [ ]:
booking_events.shape

In [ ]:
attendance.shape

In [ ]:
booking_events.info()

In [ ]:
booking_events.head(3)

In [ ]:
attendance.info()

In [ ]:
attendance.head(3)

## 4. Data quality

This section examines missing values, potential duplicate records, and basic consistency within and between the two datasets.

### Missing values

In [ ]:
booking_events.isna().sum()

In [ ]:
attendance.isna().sum()

In [ ]:
attendance[attendance.isna().any(axis=1)]

The missing values in attendance occur in the same rows. Inspection of the original CSV files suggests that these records are remnants of legacy debugging or testing code.

### Potential duplicate records

In [ ]:
attendance.duplicated(
    subset=["studio", "course", "class_start"]
).sum()

In [ ]:
attendance.loc[
    attendance.duplicated(
        subset=["studio", "course", "class_start"],
        keep=False,
    )
].sort_values(["studio", "course", "class_start"])

One duplicate attendance record was found. It originates from June 2020, predating the booking event dataset, and appears to be a legacy test record.

In [ ]:
booking_events.duplicated(
	 subset=["studio", "course", "class_start", "event_timestamp"]
).sum()

In [ ]:
booking_events.loc[
    booking_events.duplicated(
        subset=["studio", "course", "class_start", "event_timestamp"],
        keep=False,
    )
].sort_values(["studio", "course", "class_start", "event_timestamp"])

Booking events sharing the same timestamp were further inspected. Some groups showed different booking states, indicating that identical timestamps do not necessarily represent duplicated events.

### Are any events recorded after class start?

In [ ]:
(booking_events["event_timestamp"] > booking_events["class_start"]).sum()

### Do any attendance lists contain duplicate member IDs?

In [ ]:
booking_events["attendance_list"].apply(
    lambda x: len(set(x)) != len(x)
).sum()

### Do any waiting lists contain duplicate member IDs?

In [ ]:
booking_events["waiting_list"].apply(
	lambda x: len({member["ID"] for member in x}) != len(x)
).sum()

### Do any members appear in both the attendance and waiting lists?

In [ ]:
booking_events.apply(
    lambda row: bool(
        set(row["attendance_list"])
        & {member["ID"] for member in row["waiting_list"]}
    ),
    axis=1,
).sum()

In [ ]:
pd.set_option("display.max_colwidth", None)

overlap_mask = booking_events.apply(
    lambda row: bool(
        set(row["attendance_list"])
        & {member["ID"] for member in row["waiting_list"]}
    ),
    axis=1,
)

booking_events.loc[
    overlap_mask,
    ["studio", "course", "class_start", "event_timestamp",
     "attendance_list", "waiting_list"]
]

Nine records contain members who appear in both the attendance and waiting lists. Manual inspection indicates that these cases likely correspond to waitlisted members who were notified of an available spot, attended the class without updating their booking, and were subsequently added to the attendance list by the instructor without being removed from the waiting list. These records are therefore retained as valid observations.

### Can classes be consistently matched between the two datasets?

In [ ]:
class_columns = ["studio","course","class_start"]
attendance_classes = attendance[class_columns].drop_duplicates()
events_classes = booking_events[class_columns].drop_duplicates()

merged_classes = attendance_classes.merge(
	events_classes,
	on=class_columns,
	how="outer",
	indicator=True
)

merged_classes["_merge"].value_counts()


The datasets show strong referential consistency. Of the classes recorded in the booking-events dataset, 804 can be matched to a class in the attendance dataset using studio, course, and class start time. Only 7 booking-event classes have no corresponding attendance record. Manual inspection indicates that these were classes that were cancelled after booking activity had already occurred and therefore never received a final attendance record. The large number of classes appearing only in the attendance dataset is expected because the attendance data spans several years, whereas booking events are available for only about one year.

## 5. Descriptive analysis

This section summarizes the main characteristics of both datasets using descriptive statistics and simple aggregations.

### Date ranges

In [ ]:
booking_events["class_date"].min(), booking_events["class_date"].max()

In [ ]:
attendance["class_date"].min(), attendance["class_date"].max()

### Number of studios

In [ ]:
attendance["studio"].value_counts()

### Number of classes per course

In [ ]:
attendance["course"].value_counts()

### Number of unique course names

In [ ]:
attendance["course"].nunique()

### Number of classes per instructor

In [ ]:
attendance["instructor"].value_counts()

### Distribution of class capacity

In [ ]:
attendance["capacity"].describe()

### Distribution of the number of attendees

In [ ]:
attendance["attendance_length"]=attendance["attendance_list"].map(len)
attendance["attendance_length"].describe()

In [ ]:
attendance["attendance_length"].value_counts().sort_index()

### Number of booking snapshots per class

In [ ]:
booking_events.groupby(
    ["studio","course","class_start"]
).size()

## 6. Visual exploration

This section provides a visual overview of the most important characteristics of both datasets and highlights patterns relevant for the subsequent prediction task.

In [ ]:
from src.data_analysis_processing import (
    prepare_booking_events_analysis,
    prepare_attendance_analysis,
)

attendance_analysis = prepare_attendance_analysis(attendance)
booking_analysis = prepare_booking_events_analysis(booking_events)

### Distribution of class attendance

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(
    attendance_analysis["attendance_count"],
    bins=range(
        attendance_analysis["attendance_count"].min(),
        attendance_analysis["attendance_count"].max() + 2,
    ),
    edgecolor="black",
)

ax.set(
    title="Distribution of Class Attendance",
    xlabel="Number of Attendees",
    ylabel="Number of Classes",
)

plt.show()

Most classes have between 3 and 7 attendees. The distribution is left-skewed.

### Distribution of waiting list size

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

waiting_counts = (
    attendance_analysis["waiting_count"]
    .value_counts()
    .sort_index()
)

ax.bar(
    waiting_counts.index,
    waiting_counts.values,
)

ax.set(
    title="Distribution of Waiting-List Size",
    xlabel="Number of People on Waiting List",
    ylabel="Number of Classes",
)

plt.show()

### How many classes had at least one person on the waiting list?

In [ ]:
classes_with_waiting_list = (
    attendance_analysis["waiting_count"] > 0
)

print(
    f"Classes with a waiting list: "
    f"{classes_with_waiting_list.sum():,}"
)

print(
    f"Share of classes with a waiting list: "
    f"{classes_with_waiting_list.mean():.1%}"
)

### Distribution of class occupancy rate

In [ ]:
valid_occupancy = attendance_analysis.loc[
    attendance_analysis["capacity"] > 0,
    "occupancy_rate",
]

fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(
    valid_occupancy,
    bins=20,
    edgecolor="black",
)

ax.axvline(
    1.0,
    linestyle="--",
    label="Full capacity",
)

ax.set(
    title="Distribution of Class Occupancy",
    xlabel="Attendance / Capacity",
    ylabel="Number of Classes",
)

ax.legend()
plt.show()

### Which classes had more attendees than capacity allowed for?

In [ ]:
attendance_analysis.loc[
    attendance_analysis["occupancy_rate"] > 1,
    [
        "studio",
        "course",
        "class_start",
        "capacity",
        "attendance_count",
        "occupancy_rate",
    ],
].sort_values("occupancy_rate", ascending=False)

### Attendance vs. class capacity

In [ ]:
import numpy as np
fig, ax = plt.subplots(figsize=(8, 6))

hist = ax.hist2d(
    attendance_analysis["capacity"],
    attendance_analysis["attendance_count"],
    bins=[
        np.arange(3.5, 14.5, 1),
        np.arange(-0.5, 13.5, 1),
    ],
    cmap="Blues",
)

ax.plot(
    [0, 13],
    [0, 13],
    "--",
    color="black",
    label="Attendance equals capacity",
)

fig.colorbar(hist[3], ax=ax, label="Number of classes")

ax.set(
    xlabel="Class Capacity",
    ylabel="Actual Attendance",
    title="Attendance Relative to Class Capacity",
)

ax.legend()

plt.show()

In [ ]:
monthly_attendance = (
    attendance_analysis
    .set_index("class_start")
    .resample("MS")
    .agg(
        mean_attendance=("attendance_count", "mean"),
        median_attendance=("attendance_count", "median"),
        number_of_classes=("attendance_count", "size"),
    )
)

### Monthly attendance over time

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    monthly_attendance.index,
    monthly_attendance["mean_attendance"],
    label="Mean attendance",
)

ax.plot(
    monthly_attendance.index,
    monthly_attendance["median_attendance"],
    label="Median attendance",
)

ax.set(
    title="Monthly Attendance Over Time",
    xlabel="Month",
    ylabel="Attendees per Class",
)
from datetime import datetime


ax.legend()
plt.show()

### Attendance by weekday

In [ ]:
weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]

weekday_attendance = (
    attendance_analysis
    .groupby("weekday")["attendance_count"]
    .agg(["mean", "median", "count"])
    .reindex(weekday_order)
    .dropna()
)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(
    weekday_attendance.index,
    weekday_attendance["mean"],
)

ax.set(
    title="Mean Attendance by Weekday",
    xlabel="Weekday",
    ylabel="Mean Number of Attendees",
)

ax.tick_params(axis="x", rotation=45)
plt.show()

### Attendance by class start hour

In [ ]:
hourly_attendance = (
    attendance_analysis
    .groupby("class_hour")["attendance_count"]
    .agg(["mean", "median", "count"])
    .sort_index()
)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    hourly_attendance.index,
    hourly_attendance["mean"],
    marker="o",
)

ax.set(
    title="Mean Attendance by Class Start Hour",
    xlabel="Class Start Hour",
    ylabel="Mean Number of Attendees",
)

ax.set_xticks(hourly_attendance.index)
plt.show()

### Attendance by course

In [ ]:
course_summary = (
    attendance_analysis
    .groupby("course")["attendance_count"]
    .agg(
        mean_attendance="mean",
        median_attendance="median",
        number_of_classes="size",
    )
)

course_summary = course_summary.loc[
    course_summary["number_of_classes"] >= 20
].sort_values(
    "mean_attendance",
    ascending=True,
)

In [ ]:
fig, ax = plt.subplots(
    figsize=(9, max(5, len(course_summary) * 0.35))
)

ax.barh(
    course_summary.index,
    course_summary["mean_attendance"],
)

ax.set(
    title="Mean Attendance by Course",
    xlabel="Mean Number of Attendees",
    ylabel="Course",
)

plt.show()

### Distribution of number of booking snapshots per class

In [ ]:
class_columns = [
    "studio",
    "course",
    "class_start",
]

In [ ]:
snapshots_per_class = (
    booking_analysis
    .groupby(class_columns)
    .size()
    .rename("snapshot_count")
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(
    snapshots_per_class,
    bins=30,
    edgecolor="black",
)

ax.set(
    title="Number of Booking Snapshots per Class",
    xlabel="Number of Snapshots",
    ylabel="Number of Classes",
)

plt.show()

### Distribution of number of booking events by time before class

In [ ]:
booking_analysis.loc[
    booking_analysis["hours_before_class"] < 0,
    [
        "studio",
        "course",
        "class_start",
        "event_timestamp",
        "hours_before_class",
    ],
]

In [ ]:
booking_window = booking_analysis.loc[
    booking_analysis["hours_before_class"].between(0, 14 * 24)
].copy()

In [ ]:
booking_window["days_before_class_bin"] = pd.cut(
    booking_window["days_before_class"],
    bins=[0, 1, 2, 3, 5, 7, 10, 14],
    include_lowest=True,
)

In [ ]:
events_by_lead_time = (
    booking_window["days_before_class_bin"]
    .value_counts(sort=False)
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(
    events_by_lead_time.index.astype(str),
    events_by_lead_time.values,
)

ax.set(
    title="Booking Events by Time Before Class",
    xlabel="Days Before Class",
    ylabel="Number of Booking Events",
)

ax.tick_params(axis="x", rotation=45)
plt.show()

### Average number of booking snapshots by time before class

In [ ]:
from src.data_analysis_processing import create_booking_trajectory  

prediction_horizons = [
    168,
    120,
    72,
    48,
    24,
    12,
    6,
    1,
]

booking_trajectory = create_booking_trajectory(
    booking_events=booking_events,
    horizons_hours=prediction_horizons,
    class_columns=class_columns
)

In [ ]:
mean_booking_trajectory = (
    booking_trajectory
    .groupby("horizon_hours")
    .agg(
        mean_booked_count=("booked_count", "mean"),
        median_booked_count=("booked_count", "median"),
        number_of_classes=("booked_count", "size"),
    )
    .sort_index(ascending=False)
)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    mean_booking_trajectory.index,
    mean_booking_trajectory["mean_booked_count"],
    marker="o",
)

ax.invert_xaxis()

ax.set(
    title="Mean Booking Count Before Class Start",
    xlabel="Hours Before Class",
    ylabel="Mean Number of Bookings",
)

plt.show()

### Booking trajectories for a sample of classes 

In [ ]:
sample_classes = (
    booking_trajectory[
        ["studio", "course", "class_start"]
    ]
    .drop_duplicates()
    .sample(
        n=min(
            10,
            booking_trajectory[
                ["studio", "course", "class_start"]
            ]
            .drop_duplicates()
            .shape[0],
        ),
        random_state=42,
    )
)

In [ ]:
sample_trajectory = booking_trajectory.merge(
    sample_classes,
    on=["studio", "course", "class_start"],
    how="inner",
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for _, class_data in sample_trajectory.groupby(
    ["studio", "course", "class_start"]
):
    class_data = class_data.sort_values(
        "horizon_hours",
        ascending=False,
    )

    ax.plot(
        class_data["horizon_hours"],
        class_data["booked_count"],
        alpha=0.6,
    )

ax.invert_xaxis()

ax.set(
    title="Booking Trajectories for a Sample of Classes",
    xlabel="Hours Before Class",
    ylabel="Number of Bookings",
)

plt.show()

## 7. Findings from the original data

### Overall data quality

Both datasets are generally of high quality. Only a small number of missing values and potential duplicate records were identified, and manual inspection suggests that most unusual cases can be explained by the underlying booking process or legacy data.

### Strong consistency between the datasets

Classes can be matched reliably between the two datasets using studio, course, and class start time. Of the classes represented in the booking-event data, 804 have a corresponding attendance record. The seven unmatched classes were manually identified as cancelled classes for which booking activity had already occurred but no final attendance was recorded.

### Complementary temporal coverage

The attendance dataset spans several years, whereas detailed booking-event data are available for approximately one year. The attendance data therefore provide a longer historical record, while the booking-event data capture the detailed evolution of bookings before individual classes.

### Substantial variation in attendance and occupancy

Final attendance varies substantially across classes and courses. Class capacity alone does not explain this variation, suggesting that additional information about the class, booking state, and booking history may be useful for prediction.

### Booking activity contains temporal structure

Booking activity is distributed unevenly over the days preceding a class, and individual classes show different booking trajectories. The event snapshots therefore contain temporal information that would be lost if only the latest number of bookings were considered.